# EDA: player style embeddings (fixture output, not a result)

Tables from `analytics/embeddings/eda.py` on the synthetic season of `analytics.fixtures` (write_fixture_root: league rosters whose positions and aggregates share the tracked players' ids). Every number here is fixture output, not a result; the real run is the Mac command in `reports/embeddings.md`. No plots: tables and markdown only.

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd() if (Path.cwd() / 'analytics').exists() else Path.cwd().parent
sys.path.insert(0, str(REPO))
import pandas as pd
from analytics.embeddings import eda, features as F, train as T
pd.set_option('display.width', 160); pd.set_option('display.max_rows', 120)

## Inputs

The fixture season (6 games, every third held out) is written under a temporary directory and loaded through the same loader as the real data (`train.load_inputs`), so the held-out games are set aside before any feature. The fixture gate is 20 possessions.

In [ ]:
inputs = T.fixture_inputs(seed=0, n_games=6, heldout_every=3)
print(inputs['source'])
print('training games:', [g['sportvu_game'] for g in inputs['games']])
print('dropped:', inputs['dropped'])
feats = T.build_features(inputs, n_min=T.FIXTURE_N_MIN)
positions = inputs['positions']
feats[['player_id', 'season', 'n_possessions', 'insufficient']].describe(include='all')

## Feature distributions by position

Mean, sd and quantiles of every available feature per position (G, F, C) over the sufficient rows. With planted archetypes the guards carry the ball more and shoot from further out, the centres sit near the rim.

In [ ]:
dist = eda.feature_distribution_by_position(feats, positions)
dist.pivot(index='feature', columns='position', values='mean').round(3)

In [ ]:
dist[dist['feature'].isin(['ball_share', 'avg_speed_ftps', 'shot_share_3pt', 'zone_near_left', 'agg_drives'])].round(3)

## Correlation table

The twenty strongest absolute pairwise correlations between features.

In [ ]:
eda.correlation_table(feats, top=20).round(3)

## Explained variance by PCA component

In [ ]:
eda.explained_variance(feats).round(3)

## Archetype separation in the chosen embedding

`train.run` fits the candidates under the gate (PCA first; the autoencoder only when PCA's 5-NN accuracy is under 0.60). Within-label versus between-label mean distance per position and the leave-one-out 5-NN accuracy per label.

In [ ]:
ctx = T.run(inputs, seed=0, n_boot=50, n_min=T.FIXTURE_N_MIN, fixture=True)
print(ctx['decision'])
E, labels = eda.embedding_with_labels(ctx, positions)
eda.archetype_separation(E, labels).round(3)

In [ ]:
ev = ctx['candidates']['pca']['eval']
pd.DataFrame({'5-NN accuracy': ev['knn']['accuracy'], 'majority baseline': ev['knn']['majority'], 'next-season R2 (ridge)': ev['next']['r2'], 'mean baseline R2': ev['next']['r2_mean']}).round(3)

Fixture output, not a result.